# Proposal trait hero dengan NanoJev (Google Colab / Kaggle)

Notebook ini menjalankan **inference GPU** untuk membuat proposal trait berstatus `pending`. Ini bukan fine-tuning NanoJev. Tidak ada API inference, auto-approval, atau publikasi dataset. Checkpoint publik diunduh dari Hugging Face dan diverifikasi checksum-nya sebelum digunakan.

## Sebelum mulai

- Google Colab: pilih **Runtime → Change runtime type → GPU**.
- Kaggle: pilih **Settings → Accelerator → GPU** dan aktifkan Internet.
- Pastikan commit yang berisi pipeline dan data sumber terbaru sudah ada di branch GitHub yang akan dipakai di sel konfigurasi. `REPO_REF` harus menunjuk ke revision yang sama dengan checkout lokal tempat proposal nanti ditinjau.
- Jika sel batch terhenti tetapi runtime masih aktif, jalankan lagi pada direktori output yang sama; proposal valid yang sudah ada akan dilewati. Unduh ZIP hasil sebelum runtime notebook berakhir.

Setelah selesai, ekstrak file proposal dari ZIP ke `data/mappings/tactical_proposals/` di checkout lokal. Jalankan proses review lokal; jangan ubah status proposal menjadi approved sebelum memeriksa sumber dan labelnya.

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/Andndre/hok-draft-engine.git"
REPO_REF = "main"  # Ubah ke branch atau commit yang berisi sumber terbaru.

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    SESSION_DIR = pathlib.Path("/kaggle/working")
    IS_COLAB = False
elif pathlib.Path("/content").exists():
    SESSION_DIR = pathlib.Path("/content")
    IS_COLAB = True
else:
    raise RuntimeError("Jalankan notebook ini di Google Colab atau Kaggle dengan GPU aktif.")

REPO_DIR = SESSION_DIR / "hok-draft-engine"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Memakai checkout yang sudah ada: {REPO_DIR}")

ML_DIR = REPO_DIR / "packages" / "ml-engine"
PROPOSAL_DIR = REPO_DIR / "data" / "mappings" / "tactical_proposals"
CHECKPOINT_DIR = REPO_DIR / "data" / "processed" / "nanojev"
print("Repository:", REPO_DIR)
print("Revision:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import torch
from packaging.version import Version

if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA tidak terlihat. Aktifkan GPU accelerator pada runtime lalu restart sesi.")
if torch.version.cuda is None:
    raise RuntimeError("PyTorch pada runtime ini bukan build CUDA. Pilih runtime GPU dengan PyTorch CUDA.")
if Version(torch.__version__.split("+")[0]) < Version("2.15.0"):
    raise RuntimeError(f"Pipeline memerlukan PyTorch >=2.15.0 dengan CUDA; runtime menyediakan {torch.__version__}. Pilih runtime yang lebih baru.")

GPU_NAME = torch.cuda.get_device_name(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
print(f"GPU: {GPU_NAME} | VRAM kosong {free_bytes / 1024**3:.1f} / {total_bytes / 1024**3:.1f} GiB")
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp32"
print("Precision inference:", PRECISION)

# Pertahankan build PyTorch CUDA yang disediakan runtime; pasang hanya dependensi model.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==5.17.0", "safetensors==0.8.0", "huggingface-hub"], check=True)
print("Dependensi NanoJev siap.")

## Unduh checkpoint terverifikasi

Sel berikut mengunduh bobot rilis NanoJev yang dipatok di source repo. Bobot berukuran sekitar 2,4 GB; jangan masukkan checkpoint ke ZIP proposal atau Git.

In [ ]:
download_cmd = [
    sys.executable, str(ML_DIR / "tactical_profiles.py"), "download",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
]
subprocess.run(download_cmd, cwd=ML_DIR, check=True)
print("Checkpoint siap; checksum rilis sudah diverifikasi.")

## Jalankan batch hero Global

Batch hanya memilih hero yang berstatus rilis Global dan memiliki bukti sumber. Proposal disimpan per hero dengan hash sumber. Profil yang belum punya bukti atau yang gagal akan tercatat di `coverage_report.json`.

In [ ]:
PROPOSAL_DIR.mkdir(parents=True, exist_ok=True)
report_path = PROPOSAL_DIR / "coverage_report.json"
batch_cmd = [
    sys.executable, str(ML_DIR / "tactical_profiles.py"), "propose-batch",
    "--output-dir", str(PROPOSAL_DIR),
    "--report", str(report_path),
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--precision", PRECISION,
]
print("Menjalankan:", " ".join(batch_cmd))
batch_result = subprocess.run(batch_cmd, cwd=ML_DIR)
print("Kode keluar batch:", batch_result.returncode)
if report_path.exists():
    report = __import__("json").loads(report_path.read_text(encoding="utf-8"))
    print("Ringkasan:", report.get("summary", {}))
if batch_result.returncode != 0:
    print("Batch tidak lengkap. Proposal parsial dan coverage_report tetap tersedia; perbaiki penyebabnya lalu jalankan ulang sel ini.")
else:
    print("Batch selesai. Semua file yang dibuat masih berstatus pending.")

In [ ]:
import json
import zipfile

run_info = {
    "repository": REPO_URL,
    "revision": subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip(),
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "precision": PRECISION,
    "batch_exit_code": batch_result.returncode,
}
run_info_path = PROPOSAL_DIR / "run_info.json"
run_info_path.write_text(json.dumps(run_info, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
archive_path = SESSION_DIR / "nanojev_tactical_proposals.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PROPOSAL_DIR.glob("*.json")):
        archive.write(path, arcname=path.name)
print(f"ZIP proposal tersimpan: {archive_path} ({archive_path.stat().st_size / 1024**2:.1f} MiB)")
if IS_COLAB:
    from google.colab import files
    files.download(str(archive_path))
else:
    print("Kaggle: unduh nanojev_tactical_proposals.zip dari panel Output / Files pada /kaggle/working.")